In [31]:
import requests
import pandas as pd
import json

In [271]:
api_key = "ab8d15f4e5cfb79fb611f1b4228dd225"
headers ={'accept': 'application/json', 'X-API-KEY': api_key}
res = requests.get("https://api.themoviedb.org/3/search/movie", headers = headers).json()

ConnectionError: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/search/movie (Caused by NewConnectionError("HTTPSConnection(host='api.themoviedb.org', port=443): Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение"))

In [351]:
api_key2 = 'Y9YTQYE-9VXMQC9-QTHN5QV-SQQG7M8'

def more_pages(n):
    films5 = []
    for i in range (1,n+1):
        url = f'https://api.kinopoisk.dev/v1.5/movie?page={i}&limit=250&year=2022-2026&rating.kp=7-10&budget.value=1-10000000&fees.world.value=10000000-10000000000&selectFields=id&selectFields=name&selectFields=alternativeName&selectFields=fees&selectFields=type&selectFields=year&selectFields=description&selectFields=shortDescription&selectFields=movieLength&selectFields=isSeries&selectFields=ticketsOnSale&selectFields=totalSeriesLength&selectFields=seriesLength&selectFields=ageRating&selectFields=typeNumber&selectFields=status&selectFields=names&selectFields=rating&selectFields=votes&selectFields=genres&selectFields=countries&selectFields=releaseYears&selectFields=budget'
        headers ={'accept': 'application/json', 'X-API-KEY': api_key2}
        page = requests.get(url, headers=headers)
        
        response = page.json()
        res = response.get('docs')
        films5.extend(res)

        time.sleep(0.5)

    return films5

In [352]:
movies = more_pages(60)

In [356]:
films = pd.DataFrame(movies)

In [357]:
films['kp_votes'] = films['votes'].apply(lambda d: d['kp'])
films['kp_rating'] = films['rating'].apply(lambda d: d['kp'])

films = films.drop('rating',axis = 1)
films = films.drop('votes',axis = 1)

films['genre'] = films['genres'].fillna('').apply(lambda genres: ', '.join([genre['name'] for genre in genres]))
films['country'] = films['countries'].fillna('').apply(lambda countries: ', '.join([country['name'] for country in countries]))

films = films.drop('countries',axis = 1)
films = films.drop('genres',axis = 1)

films = films.drop('names',axis = 1)

In [366]:
films['film_budget'] = films['budget'].apply(lambda d: d['value'])
films['film_fees'] = films['fees'].apply(lambda d: d['world']).apply(lambda d: d['value'])

In [369]:
movies1 = films[['name','year','type','kp_votes','kp_rating','genre','country','film_budget','film_fees']]

In [371]:
movies1

,name,year,type,kp_votes,kp_rating,genre,country,film_budget,film_fees
0,Пропавшая без вести,2023,movie,32646,7.202,"детектив, триллер, драма",США,7000000,48767848
1,Криминальный город 2,2022,movie,150316,7.600,"криминал, боевик, триллер, детектив",Корея Южная,9000000,106247455
2,Ужастики. Ожившие рисунки,2024,movie,35143,7.406,"фэнтези, комедия, семейный",США,4800000,10253737
3,Национальное достояние,2025,movie,701,7.367,драма,Япония,8000000,134438321
4,Махараджа,2024,movie,1981,7.114,"боевик, триллер, драма, криминал",Индия,2500000,21500000
...,...,...,...,...,...,...,...,...,...
1375,Кит,2022,movie,78726,7.650,драма,США,10000000,57026694
1376,Решение уйти,2022,movie,62549,7.080,"детектив, триллер, мелодрама, криминал",Корея Южная,10000000,23095762
1377,6 из 45,2022,movie,872,7.516,комедия,Корея Южная,4000000,23335860
1378,Пожарные,2024,movie,12659,7.349,"драма, боевик",Корея Южная,8500000,25223079


In [373]:
movies1.to_csv('kinopoisk_data.csv', index = False)